# Loki stocks pilot on WRDS
This notebook runs in WRDS JupyterHub. No live WRDS result is bundled. The initial June 2024 window is an example, not agreed final coverage. Use a reviewed source checkout containing src/stocks. This is a fresh-pilot template; use docs/stocks_history_repro.md to audit existing history without rebuilding it.

All licensed source/output rows stay under the WRDS data root. The ten pilot labels come from `config/universe.csv`. The source is legacy CRSP daily data; schema mismatch stops the query. For large SSH jobs use WRDS's scheduled compute-node workflow.

In [ ]:
from pathlib import Path
import os, sys

# Open from the extracted project's notebooks/ folder or project root.
project_root = Path.cwd()
if not (project_root / 'src' / 'stocks').is_dir():
    project_root = project_root.parent
if not (project_root / 'src' / 'stocks').is_dir():
    raise RuntimeError('Open this notebook inside the extracted course project')
os.chdir(project_root)
# Kernel restart may retain an absolute notebooks/ entry rather than the new cwd.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('Project:', project_root)
print('Python:', sys.version)


Check the packages already available in the WRDS kernel. Do not install or upgrade packages by default: the observed WRDS session cannot reach PyPI. python-dotenv is optional in shared config; the username can be entered later. The local pins target Python 3.11.9. The separate WRDS Python 3.14.5 run used preinstalled packages; see docs/stocks_validation_2026-10-23.md. Do not force local pins into that kernel.


In [ ]:
import importlib
import sys

print('Python:', sys.version.split()[0])
missing_runtime = []
for name in ['wrds', 'pandas', 'numpy']:
    try:
        module = importlib.import_module(name)
        print(name, getattr(module, '__version__', 'import OK'))
    except ImportError as exc:
        missing_runtime.append(name)
        print(name, 'IMPORT FAILED:', exc)
try:
    import pytest
    print('pytest', pytest.__version__)
except ImportError:
    print('pytest: unavailable; cloud tests have not run')
if missing_runtime:
    raise RuntimeError(f'Missing runtime packages: {missing_runtime}. Stop and inspect; do not retry internet pip installation.')
from src.common import config
print('dotenv loader available:', config.load_dotenv is not None, '(optional)')
print('Pilot tickers:', config.load_universe())


In [ ]:
# Run offline tests before opening a database connection.
import importlib.util
import subprocess, tempfile
if importlib.util.find_spec('pytest') is None:
    raise RuntimeError('pytest is unavailable. Cloud tests have not run; ask for offline setup instructions before continuing.')
with tempfile.TemporaryDirectory(prefix='stocks-pytest-') as test_dir:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/stocks', '--basetemp', str(Path(test_dir) / 'cases')], check=True)


In [ ]:
from src.common import config
from src.stocks.pull import data_root, raw_folder, window

START, END = window('2024-06-01', '2024-06-30')  # Bounded starting example.
# Default: project/data on WRDS. Optionally set an approved persistent path.
DATA_ROOT = data_root()
# DATA_ROOT = Path('/your/approved/persistent/stocks-data')
print('Pilot tickers:', config.load_universe())
print('Window:', START, END)
print('Data root:', DATA_ROOT)


The next cell opens a real WRDS connection and requests actual data. Authenticate normally when WRDS prompts; do not put a password or credential in the source. If the raw run folder exists, skip the pull. Build only when processed output does not already exist; use the history audit workflow for saved processed data. An incomplete folder needs inspection; it is never overwritten or treated as a successful run.

In [ ]:
from src.common.wrds_conn import connect
from src.stocks.pull import pull

if raw_folder(DATA_ROOT, START, END).exists():
    print('Raw run exists; skipping pull. Build below will check completeness and hashes.')
else:
    if not config.WRDS_USERNAME:
        config.WRDS_USERNAME = input('WRDS username (not password): ').strip()
    db = connect()
    try:
        raw_path = pull(db, DATA_ROOT, START, END, config.load_universe())
        print('Raw saved:', raw_path)
    finally:
        db.close()


In [ ]:
from src.stocks.build import build
import json
# Protect existing processed tables and any audits stored below them.
existing_output = DATA_ROOT / 'processed' / 'stocks' / f'{START}_{END}'
if existing_output.exists():
    raise FileExistsError(
        f'Processed output already exists: {existing_output}. '
        'Use docs/stocks_history_repro.md to create a separate audit run.'
    )
output, report = build(DATA_ROOT, START, END)
print('Processed:', output)
print(json.dumps(report, indent=2))


In [ ]:
import pandas as pd
coverage = pd.read_csv(output / 'coverage.csv')
display(coverage)
daily = pd.read_csv(output / 'stock_daily.csv')
display(daily[daily['core_flags'].fillna('').ne('')].head(25))


Review every core flag and missing security before integration. The selected ten securities do not form a survivorship-free S&P universe. Observed dates do not prove calendar completeness; closing quotes do not establish executable fills. Corporate-action events, delisting-adjusted returns, broker data and OptionMetrics links remain separate work. Keep licensed rows and this notebook's data-bearing outputs on WRDS; clear outputs before sharing source.